In [ ]:
# Experiment 3: Decision Tree using ID3 Algorithm
# Aim: Demonstrate the working of the ID3 algorithm
#      and classify a new sample.

import pandas as pd
import math

# ---------------------------------------------------------
# STEP 1: Create the dataset
# ---------------------------------------------------------

data = {
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain',
                'Overcast', 'Sunny', 'Sunny', 'Rain', 'Sunny', 'Overcast',
                'Overcast', 'Rain'],

    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool',
                    'Cool', 'Mild', 'Cool', 'Mild', 'Mild', 'Mild',
                    'Hot', 'Mild'],

    'Humidity': ['High', 'High', 'High', 'High', 'Normal', 'Normal',
                 'Normal', 'High', 'Normal', 'Normal', 'Normal', 'High',
                 'Normal', 'High'],

    'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong',
             'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong',
             'Weak', 'Strong'],

    'PlayTennis': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No',
                   'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes',
                   'Yes', 'No']
}

df = pd.DataFrame(data)

print("Training Dataset:\n")
print(df)


# ---------------------------------------------------------
# STEP 2: Calculate Entropy
# ---------------------------------------------------------

def entropy(data):
    values = data.value_counts()

    total = len(data)
    ent = 0

    for count in values:
        probability = count / total
        ent -= probability * math.log2(probability)

    return ent


# ---------------------------------------------------------
# STEP 3: Calculate Information Gain
# ---------------------------------------------------------

def information_gain(df, attribute, target):
    total_entropy = entropy(df[target])

    values = df[attribute].unique()
    weighted_entropy = 0

    for value in values:
        subset = df[df[attribute] == value]
        weighted_entropy += (len(subset) / len(df)) * entropy(subset[target])

    return total_entropy - weighted_entropy


# ---------------------------------------------------------
# STEP 4: Display Information Gain
# ---------------------------------------------------------

print("\nEntropy of complete dataset:",
      round(entropy(df['PlayTennis']), 4))

print("\nInformation Gain for each attribute:")

attributes = ['Outlook', 'Temperature', 'Humidity', 'Wind']

for attribute in attributes:
    gain = information_gain(df, attribute, 'PlayTennis')
    print(attribute, "=", round(gain, 4))


# ---------------------------------------------------------
# STEP 5: Implement ID3 Algorithm
# ---------------------------------------------------------

def ID3(df, target, attributes):

    # If all examples belong to the same class
    if len(df[target].unique()) == 1:
        return df[target].iloc[0]

    # If no attributes are left
    if len(attributes) == 0:
        return df[target].mode()[0]

    # Select attribute with maximum Information Gain
    gains = {
        attribute: information_gain(df, attribute, target)
        for attribute in attributes
    }

    best_attribute = max(gains, key=gains.get)

    tree = {best_attribute: {}}

    # Create branches
    for value in df[best_attribute].unique():

        subset = df[df[best_attribute] == value]

        remaining_attributes = [
            attribute for attribute in attributes
            if attribute != best_attribute
        ]

        tree[best_attribute][value] = ID3(
            subset,
            target,
            remaining_attributes
        )

    return tree


# ---------------------------------------------------------
# STEP 6: Build the Decision Tree
# ---------------------------------------------------------

tree = ID3(
    df,
    'PlayTennis',
    ['Outlook', 'Temperature', 'Humidity', 'Wind']
)

print("\nDecision Tree:")
print(tree)


# ---------------------------------------------------------
# STEP 7: Function to classify a new sample
# ---------------------------------------------------------

def classify(tree, sample):

    if not isinstance(tree, dict):
        return tree

    attribute = next(iter(tree))

    value = sample[attribute]

    if value in tree[attribute]:
        return classify(tree[attribute][value], sample)

    return "Unknown"


# ---------------------------------------------------------
# STEP 8: Classify a New Sample
# ---------------------------------------------------------

new_sample = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

prediction = classify(tree, new_sample)

print("\nNew Sample:")
print(new_sample)

print("\nClassification Result:")
print("Play Tennis =", prediction)

Training Dataset:

     Outlook Temperature Humidity    Wind PlayTennis
0      Sunny         Hot     High    Weak         No
1      Sunny         Hot     High  Strong         No
2   Overcast         Hot     High    Weak        Yes
3       Rain        Mild     High    Weak        Yes
4       Rain        Cool   Normal    Weak        Yes
5       Rain        Cool   Normal  Strong         No
6   Overcast        Cool   Normal  Strong        Yes
7      Sunny        Mild     High    Weak         No
8      Sunny        Cool   Normal    Weak        Yes
9       Rain        Mild   Normal    Weak        Yes
10     Sunny        Mild   Normal  Strong        Yes
11  Overcast        Mild     High  Strong        Yes
12  Overcast         Hot   Normal    Weak        Yes
13      Rain        Mild     High  Strong         No

Entropy of complete dataset: 0.9403

Information Gain for each attribute:
Outlook = 0.2467
Temperature = 0.0292
Humidity = 0.1518
Wind = 0.0481

Decision Tree:
{'Outlook': {'Sunny': {'H